In [1]:
import importlib
import sys
import os
import torch
import numpy as np
from tqdm.notebook import tqdm
import torch

sys.path.insert(0, '..')
sys.path.insert(1, '../../..')
sys.path.insert(0, "../../src")  # src package


## Generate Train, Val, Test

In [ ]:
from perturbation_logic.activity_pertubator import (
    split_prefix_suffix_readable,
    redo_last_activity_of_prefix)
from event_log_loader_service.event_log_loader import (get_train_test_val_datasets,
                                                       extract_feature_info)
np.random.seed(17)

csv_path="../../../data/BPI Challenge 2017.csv"

properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name',
    'timestamp_name' : 'time:timestamp',
    'time_since_case_start_column' : 'case_elapsed_time',
    'time_since_last_event_column' : 'event_elapsed_time',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 5,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.2,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name', 'Action', 'org:resource', 'EventOrigin', 'lifecycle:transition', 'case:LoanGoal', 'case:ApplicationType', 'Accepted', 'Selected', ],
    'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost', 'CreditScore'],
    'continuous_positive_columns' : []
}


train_df, val_df, test_df,  = get_train_test_val_datasets(csv_path, properties)


print(len(train_df))

# data_train = split_prefix_suffix_readable(
#     train_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=2,
# )

data_val = split_prefix_suffix_readable(
    val_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=5,
)

len(data_val)
# data_test = split_prefix_suffix_readable(
#     test_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=2,
# )

#torch.save(data_train, '../../../perturbed_data/helpdesk/train.pkl')
#torch.save(data_val, '../../../perturbed_data/BPIC17/val.pkl')
#torch.save(data_test, '../../../perturbed_data/helpdesk/test.pkl')


#display(train_df)

#extract feature info
feature_info = extract_feature_info(val_df, properties)


881621


# Create perturbed Datasets

In [3]:
# Last Event Attack
from perturbation_logic.feature_attacks import last_event_attack

# Reset data_val_copy for feature attacks
data_val_copy = data_val.copy()

# Attacks the last event of each prefix
data_pert_last = last_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['lifecycle:transition','org:resource'],
    num_of_features_to_attack=2,
    magnitude=0.5,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_last, '../../../perturbed_data/BPIC17/last_event_attack_all.pkl')

In [4]:
# Random Event Attack
from perturbation_logic.feature_attacks import random_event_attack

# Attacks random events in each prefix with probability p
data_val_copy = data_val.copy()  
data_pert_random = random_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['lifecycle:transition','org:resource'],
    num_of_features_to_attack=2,
    event_attack_probability=1.0,
    magnitude=0.5,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_random, '../../../perturbed_data/BPIC17/random_event_attack_all.pkl')

In [ ]:
# Apply "redo last activity" augmentation to each prefix/suffix pair

data_val_copy = data_val.copy()  # Reset again
data_pert = {}
for key, (prefix_df, suffix_df) in data_val_copy.items():
    new_prefix, new_suffix = redo_last_activity_of_prefix(
        prefix_df,
        suffix_df,
        properties=properties,
    )
    data_pert[key] = (new_prefix, new_suffix)


torch.save(data_pert, '../../../perturbed_data/BPIC17/redo_pert.pkl')


In [ ]:
# Apply " loop augmentation"
from perturbation_logic.structural_attacks import generate_loop_augmentation

val_loops_clean, val_loops_pert = generate_loop_augmentation(
    val_df,
    properties,
    min_suffix_size=5,
    max_matches_per_loop=3,
    save_path="../../../perturbed_data/BPIC17",
)

# Save final results
torch.save(val_loops_clean, "../../../perturbed_data/BPIC17/loop_augmentation_clean_new.pkl")
torch.save(val_loops_pert, "../../../perturbed_data/BPIC17/loop_augmentation_pert_new.pkl")
print(f"Loop augmentation: {len(val_loops_clean)} clean/pert pairs") 

# Compare changes

In [5]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_feature_attack_impact

# Compare clean dataset with random event attack
highlight_feature_attack_impact(
    clean_data_path='../../../perturbed_data/BPIC17/val.pkl',
    perturbed_data_path='../../../perturbed_data/BPIC17/last_event_attack_all.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/BPIC17/val.pkl
Loading perturbed dataset from: ../../../perturbed_data/BPIC17/last_event_attack_all.pkl

Clean dataset has 171611 cases
Perturbed dataset has 171611 cases

COMPARISON RESULTS

Case: Application_1000086665, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    org:resource:
      Clean:    User_1
      Perturbed: User_47 [CHANGED]


Case: Application_1000086665, Prefix Length: 2
  ✓ No changes detected

Case: Application_1000086665, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    lifecycle:transition:
      Clean:    schedule
      Perturbed: withdraw [CHANGED]
    org:resource:
      Clean:    User_1
      Perturbed: User_63 [CHANGED]


Case: Application_1000086665, Prefix Length: 4
  [CHANGED] Found 1 event(s) with differences

  Event 3:
    lifecycle:transition:
      Clean:    withdraw
      Perturbed: complete [CHANGED]
    org:resource:
      Clean:    User_

KeyboardInterrupt: 

In [ ]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_structural_attack_impact

# Compare clean dataset with random event attack
highlight_structural_attack_impact(
    clean_data_path='../../../perturbed_data/BPIC17/test_clean.pkl',
    perturbed_data_path='../../../perturbed_data/BPIC17/test_redo_pert.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/helpdesk/test_clean.pkl
Loading perturbed dataset from: ../../../perturbed_data/helpdesk/test_redo_pert.pkl

Clean dataset has 2444 cases
Perturbed dataset has 2444 cases

STRUCTURAL ATTACK COMPARISON RESULTS

Case: Case 1617, Prefix Length: 1
Prefix
Activity_seq_clean = ['Assign seriousness']
Activity_seq_pert = ['Assign seriousness', 'Assign seriousness']
Case_elapsed_time_clean = [0.0]
Case_elapsed_time_pert = [0.0, 0.0]
Event_elapsed_time_clean = [nan]
Event_elapsed_time_pert = [nan, nan]
Day_in_week_clean = [3.0]
Day_in_week_pert = [3.0, 3.0]
Seconds_in_day_clean = [25523.0]
Seconds_in_day_pert = [25523.0, 25523.0]

===
suffix
Activity_seq_clean = ['Wait', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
Activity_seq_pert = ['Wait', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
Case_elapsed_time_clean = [6477.0, 2967139.0, 2967147.0, 4263168.0, nan, 